In [10]:
# Core data manipulation
import pandas as pd
import numpy as np
from numpy import polyfit, poly1d

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
import altair as alt
import plotly.express as px

In [2]:
owid = pd.read_csv('owid-covid-data.csv')
display(owid.head())

,iso_code,continent,location,date,total_cases,new_cases,new_cases_smoothed,total_deaths,new_deaths,new_deaths_smoothed,...,male_smokers,handwashing_facilities,hospital_beds_per_thousand,life_expectancy,human_development_index,population,excess_mortality_cumulative_absolute,excess_mortality_cumulative,excess_mortality,excess_mortality_cumulative_per_million
0,AFG,Asia,Afghanistan,2020-01-05,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN
1,AFG,Asia,Afghanistan,2020-01-06,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN
2,AFG,Asia,Afghanistan,2020-01-07,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN
3,AFG,Asia,Afghanistan,2020-01-08,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN
4,AFG,Asia,Afghanistan,2020-01-09,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN


In [3]:
owid['stringency_index']

0         0.0
1         0.0
2         0.0
3         0.0
4         0.0
         ... 
429430    NaN
429431    NaN
429432    NaN
429433    NaN
429434    NaN
Name: stringency_index, Length: 429435, dtype: float64

### Visualization #1

In [32]:
import pycountry
from vega_datasets import data as vega_data

deaths = (
    owid.groupby(['iso_code', 'location'])['total_deaths_per_million']
    .max()
    .reset_index()
)

# Convert ISO alpha-3 to numeric ID to match the map
def to_numeric(iso3):
    try:
        return int(pycountry.countries.get(alpha_3=iso3).numeric)
    except:
        return None

deaths['id'] = deaths['iso_code'].apply(to_numeric)
deaths = deaths.dropna(subset=['id'])
deaths['id'] = deaths['id'].astype(int)

world = alt.topo_feature(vega_data.world_110m.url, 'countries')

alt.Chart(world).mark_geoshape().encode(
    color=alt.Color(
        'total_deaths_per_million:Q',
        scale=alt.Scale(scheme='yelloworangered',reverse=False),
        title='Deaths per Million'),
    tooltip=['location:N', 'total_deaths_per_million:Q']).transform_lookup(
    lookup='id',
    from_=alt.LookupData(
        data=deaths,
        key='id',
        fields=['total_deaths_per_million', 'location']
    )).project('naturalEarth1').properties(
    width=800,
    height=450,
    title='COVID-19 Deaths per Million by Country'
)

alt.Chart(...)

### Visualization #2

In [27]:
alt.data_transformers.disable_max_rows()

# Filter to 2020-2022, monthly average
owid['date'] = pd.to_datetime(owid['date'])
stringency_monthly = (
    owid[
        (owid['date'] >= '2020-01-01') &
        (owid['date'] <= '2022-12-31') &
        owid['stringency_index'].notna()
    ]
    .groupby(['location', pd.Grouper(key='date', freq='MS')])['stringency_index']
    .mean()
    .reset_index()
)

highlight = ['United States', 'China', 'New Zealand', 'Sweden', 'Brazil', 'Germany']

# --- Background grey lines (all other countries) ---
grey = alt.Chart(
    stringency_monthly[~stringency_monthly['location'].isin(highlight)]).mark_line(opacity=0.08, color='grey', strokeWidth=0.8).encode(
    x=alt.X('date:T', title='', axis=alt.Axis(format='%b %Y', tickCount={'interval': 'month', 'step': 3})),
    y=alt.Y('stringency_index:Q', title='Stringency Index', scale=alt.Scale(domain=[0, 100])),
    detail='location:N'
)

# --- Highlighted country lines ---
color_scale = alt.Scale(
    domain=highlight,
    range=['#1f77b4', '#d62728', '#2ca02c', '#aaaaaa', '#ff7f0e', '#9467bd']
    # US=blue, China=red, NZ=green, Sweden=grey, Brazil=orange, Germany=purple
)

lines = alt.Chart(
    stringency_monthly[stringency_monthly['location'].isin(highlight)]).mark_line(strokeWidth=2.2).encode(
    x='date:T',
    y='stringency_index:Q',
    color=alt.Color('location:N', scale=color_scale, title='Country'),
    tooltip=[
        alt.Tooltip('location:N', title='Country'),
        alt.Tooltip('date:T', title='Date', format='%b %Y'),
        alt.Tooltip('stringency_index:Q', title='Stringency Index', format='.1f')
    ]
)

# --- Key event annotations ---
events = pd.DataFrame([
    {'date': '2020-01-23', 'label': 'Wuhan Lockdown',       'y': 96},
    {'date': '2020-03-11', 'label': 'WHO Pandemic',          'y': 88},
    {'date': '2020-03-26', 'label': 'NZ Lockdown',           'y': 80},
    {'date': '2020-06-08', 'label': 'NZ Reopens',            'y': 96},
    {'date': '2021-01-20', 'label': 'US Mask Mandate',       'y': 88},
    {'date': '2021-07-01', 'label': 'Delta Wave',             'y': 80},
    {'date': '2021-12-01', 'label': 'Omicron Wave',          'y': 96},
    {'date': '2022-03-01', 'label': 'Most Restrictions Lift','y': 88},
])
events['date'] = pd.to_datetime(events['date'])

annotation_rules = alt.Chart(events).mark_rule(
    strokeDash=[4, 4], opacity=0.4, color='black', strokeWidth=1).encode(x='date:T')

annotation_text = alt.Chart(events).mark_text(
    align='left', dx=4, fontSize=9, color='#333333', fontStyle='italic').encode(
    x='date:T',
    y=alt.Y('y:Q'),
    text='label:N')

# --- Combine ---
chart = (grey + lines + annotation_rules + annotation_text).properties(
    width=850,
    height=480,
    title=alt.TitleParams(
        'COVID-19 Oxford Stringency Index by Country (2020–2022)',
        fontSize=15
    )).configure_axis(
    grid=True, gridOpacity=0.2).configure_view(
    strokeWidth=0
)

chart

alt.LayerChart(...)

### Visualization #3

In [4]:
# setting date to date time format
owid['date'] = pd.to_datetime(owid['date'])

# ---- 1️⃣ First confirmed death per country ----
first_death = (
    owid[owid['total_deaths'] > 0]
    .groupby('location')['date']
    .min()
    .reset_index()
    .rename(columns={'date': 'first_death_date'})
)

# ---- 2️⃣ First strong restriction date (stringency ≥ 70) ----
strong_restriction = (
    owid[owid['stringency_index'] >= 70]
    .groupby('location')['date']
    .min()
    .reset_index()
    .rename(columns={'date': 'restriction_date'})
)

# ---- 3️⃣ Merge these together ----
country_dates = pd.merge(first_death, strong_restriction, on='location', how='inner')

# ---- 4️⃣ Calculate days between ----
country_dates['days_to_restrictions'] = (
    country_dates['restriction_date'] - country_dates['first_death_date']
).dt.days

# ---- 5️⃣ Get deaths per million (use max value per country) ----
deaths_pm = (
    owid.groupby('location')['total_deaths_per_million']
    .max()
    .reset_index()
)

# ---- 6️⃣ Get continent + population ----
meta = (
    owid.groupby('location')
    .agg({
        'continent': 'first',
        'population': 'first'
    })
    .reset_index()
)

# ---- 7️⃣ Final dataset ----
country_owid = (
    country_dates
    .merge(deaths_pm, on='location')
    .merge(meta, on='location')
)

# Drop missing continent rows (e.g., World, income groups)
country_owid = country_owid[country_owid['continent'].notna()]

country_owid.head()

,location,first_death_date,restriction_date,days_to_restrictions,total_deaths_per_million,continent,population
0,Afghanistan,2020-03-29,2020-04-05,7,197.10,Asia,41128772
1,Albania,2020-03-15,2020-03-13,-2,1274.93,Europe,2842318
2,Algeria,2020-03-29,2020-03-23,-6,151.31,Africa,44903228
3,Angola,2020-05-24,2020-03-27,-58,54.36,Africa,35588996
4,Argentina,2020-03-08,2020-03-19,11,2877.54,South America,45510324


In [5]:
country_owid = country_owid[
    country_owid['days_to_restrictions'].notna() &
    country_owid['total_deaths_per_million'].notna() &
    country_owid['population'].notna()
]

In [6]:
country_owid[['days_to_restrictions',
            'total_deaths_per_million',
            'population']].isna().sum()

days_to_restrictions        0
total_deaths_per_million    0
population                  0
dtype: int64

In [7]:
alt.data_transformers.disable_max_rows()

scatter = alt.Chart(country_owid).mark_circle(opacity=0.7).encode(
    x=alt.X('days_to_restrictions:Q',
            title='Days Between First Death and Strong Restrictions'),
    y=alt.Y('total_deaths_per_million:Q',
            title='COVID Deaths per Million'),
    color=alt.Color('continent:N', title='Continent'),
    size=alt.Size('population:Q',
                  scale=alt.Scale(type='sqrt', range=[30, 2000]),
                  legend=alt.Legend(title='Population')),
    tooltip=['location', 'continent',
             'days_to_restrictions',
             'total_deaths_per_million',
             'population']
).properties(
    width=700,
    height=500,
    title='Policy Timing and COVID Mortality'  
)

# regression = alt.Chart(country_owid).transform_regression(
#     'days_to_restrictions',
#     'total_deaths_per_million'
# ).mark_line(color='black')

# chart = (scatter + regression).properties(
#     width=700,
#     height=500,
#     title='Policy Timing and COVID Mortality'
# )

scatter

alt.Chart(...)

In [9]:
alt.data_transformers.disable_max_rows()

scatter = alt.Chart(country_owid).mark_circle(opacity=0.7).encode(
    x=alt.X(
        'days_to_restrictions:Q',
        title='Days Between First Death and Strong Restrictions'
    ),
    y=alt.Y(
        'total_deaths_per_million:Q',
        title='COVID Deaths per Million'
    ),
    color=alt.Color('continent:N', title='Continent'),
    size=alt.Size(
        'population:Q',
        scale=alt.Scale(range=[50, 3000]),
        title='Population'
    ),
    tooltip=[
        'location',
        'continent',
        'days_to_restrictions',
        'total_deaths_per_million',
        'population'
    ]
)

# Regression line
regression = alt.Chart(country_owid).transform_regression(
    'days_to_restrictions',
    'total_deaths_per_million'
).mark_line(color='black')

# Label important countries
important = ['United States', 'Italy', 'China', 'Brazil', 'India']

labels = alt.Chart(
    country_owid[country_owid['location'].isin(important)]
).mark_text(
    align='left',
    dx=7,
    dy=-5
).encode(
    x='days_to_restrictions:Q',
    y='total_deaths_per_million:Q',
    text='location'
)

chart = (scatter + regression + labels).properties(
    width=700,
    height=500,
    title='Policy Timing and COVID Mortality'
)

chart

alt.LayerChart(...)